In [1]:
import numpy as np
import pandas as pd
import sklearn
import warnings

warnings.filterwarnings("ignore")

In [4]:
df = pd.read_csv('..\data\processed\df_quality_score.csv')
df.head()

,Age,Years_Experience,WFH_Days_Per_Week,Gender,Education_Level,Marital_Status,Has_Children,Location_Type,Department,Job_Level,...,Team_Collaboration_Frequency,Quality_Score,Innovation_Score,Efficiency_Rating,Meetings_Per_Week,Commute_Time_Minutes,Job_Satisfaction,Stress_Level,Work_Life_Balance,Smart_Work_Index
0,39,10,2,Female,Associate Degree,Married,Yes,Urban,Product,Mid-Level,...,Few times per week,58.1,52.1,72.1,4,48,55.9,6,8,2948.86
1,33,4,5,Female,Master Degree,Married,No,Urban,Customer Success,Senior,...,Monthly,93.3,77.9,89.5,12,0,96.1,3,8,5515.32
2,40,3,3,Male,PhD,Single,Yes,Rural,Operations,Mid-Level,...,Few times per week,84.7,63.2,95.0,15,24,90.4,5,6,5176.08
3,48,14,3,Male,Bachelor Degree,Married,Yes,Urban,Finance,Manager,...,Daily,67.8,82.5,95.0,8,8,100.0,10,5,5791.50
4,32,6,5,Male,High School,Divorced,Yes,Rural,Engineering,Senior,...,Few times per week,86.4,67.5,95.0,10,0,100.0,3,4,6628.50


In [6]:
X = df.drop('Quality_Score', axis=1)
y = df['Quality_Score']
print(X.shape, y.shape)

(1440, 25) (1440,)


In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=X['Gender'])

In [27]:
cat_cols = X_test.select_dtypes(['object','category']).columns.tolist()
num_cols = X_test.drop(cat_cols, axis=1).columns.tolist()
type(num_cols), type(cat_cols)

(list, list)

In [28]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

In [29]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
X_train_processed.shape, X_test_processed.shape

((1152, 78), (288, 78))

In [33]:
from sklearn.linear_model import LinearRegression,Ridge,Lasso,ElasticNet
from sklearn.ensemble import RandomForestRegressor,HistGradientBoostingRegressor,GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor  

from sklearn.metrics import mean_absolute_error, r2_score,root_mean_squared_error
import time

In [34]:
modellar ={
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'ElasticNet Regression': ElasticNet(),
    'Random Forest Regressor': RandomForestRegressor(),
    'Gradient Boosting Regressor': GradientBoostingRegressor(),
    'Hist Gradient Boosting Regressor': HistGradientBoostingRegressor(),
    'K-Nearest Neighbors Regressor': KNeighborsRegressor(),
    'Support Vector Regressor': SVR(),
    'Decision Tree Regressor': DecisionTreeRegressor(),
    'MLP Regressor': MLPRegressor(max_iter=500,random_state=42),
    
    'XGBoost Regressor': XGBRegressor(objective='reg:squarederror', eval_metric='rmse'),
    'LightGBM Regressor': LGBMRegressor(objective='regression', metric='rmse',verbose=-1),
    'CatBoost Regressor': CatBoostRegressor(verbose=0, objective='RMSE')
}

In [38]:
natijalar = []

for nomi, model in modellar.items():
    print(f"⏳ {nomi} o'qitilmoqda...")
    start_time = time.time()
    
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    lag_time = time.time() - start_time
    
    natijalar.append({
        'Model': nomi,
        'MAE': round(mae, 3),
        'RMSE': round(rmse, 3),
        'R2 Score': round(r2, 3),
        'Ketgan vaqt (sek)': round(lag_time, 3)
    })
    
df_natija = pd.DataFrame(natijalar)

⏳ Linear Regression o'qitilmoqda...
⏳ Ridge Regression o'qitilmoqda...
⏳ Lasso Regression o'qitilmoqda...
⏳ ElasticNet Regression o'qitilmoqda...
⏳ Random Forest Regressor o'qitilmoqda...
⏳ Gradient Boosting Regressor o'qitilmoqda...
⏳ Hist Gradient Boosting Regressor o'qitilmoqda...
⏳ K-Nearest Neighbors Regressor o'qitilmoqda...
⏳ Support Vector Regressor o'qitilmoqda...
⏳ Decision Tree Regressor o'qitilmoqda...
⏳ MLP Regressor o'qitilmoqda...
⏳ XGBoost Regressor o'qitilmoqda...
⏳ LightGBM Regressor o'qitilmoqda...
⏳ CatBoost Regressor o'qitilmoqda...


In [39]:
df_natija.sort_values(by='R2 Score', ascending=False)

,Model,MAE,RMSE,R2 Score,Ketgan vaqt (sek)
4,Random Forest Regressor,5.177,6.578,0.751,0.955
5,Gradient Boosting Regressor,5.275,6.620,0.748,0.281
13,CatBoost Regressor,5.193,6.637,0.746,1.473
12,LightGBM Regressor,5.335,6.799,0.734,0.110
1,Ridge Regression,5.472,6.954,0.722,0.004
6,Hist Gradient Boosting Regressor,5.435,6.953,0.722,0.256
0,Linear Regression,5.474,6.966,0.721,0.015
2,Lasso Regression,5.608,6.958,0.721,0.001
10,MLP Regressor,5.454,6.993,0.718,1.556
11,XGBoost Regressor,5.731,7.207,0.701,0.086
